# Tutorial 4.1b: One-hour-ahead PV point forecasting

Use **GEFCom2014 solar Zone 3**, an Australian PV series independent of the building
in Part a. Predict normalised PV power in p.u. one hour ahead using the previous
PV observation and weather forecasts for the target hour. This notebook runs independently.

Check when predictors are available, fit a linear model, apply prediction
bounds, and compare persistence, regression, a decision tree and a random forest. Reserve calibration data for the next session.
Tutorial 4.2 uses linear regression to examine forecast uncertainty;
its model choice is specified before test evaluation.

Run from the repository root. Allow **35–45 minutes**: setup/availability 10,
features/models/corrections 20, scoring/interpretation 10, calibration/test discussion 5 minutes.
Complete six short TODO cells, including fitting all three models. Probabilistic forecasts are covered on Day 2.
Source: [Zone 3 extract](data/tutorial_4_2/README.md) and [Day 1 preparation](data/week4_day1/README.md).

## P0. Load the data and define chronological splits (slide 31)

The data include forecast irradiance (W/m²), forecast air temperature (°C),
forecast cloud cover (fraction) and observed PV power (p.u.).
Retain the weather forecast's issue time and valid time. Irradiance has already been
calculated from differences in accumulated radiation, divided by 3600 seconds.
Do not difference this column again. All timestamps are in **UTC**.
The exact location is undisclosed, so 12:00 UTC should not be described as local midday.

One source value slightly exceeds 1 p.u. Preserve the original values in
`solar_power_pu_raw`, and create a separate target constrained to [0,1] before calculating
lags. Report the values changed by this bound. This bound is a modelling choice for the exercise:
normalised output can exceed nominal capacity in the original data.

Training: 1 April 2012 01:00–31 March 2013 23:00; calibration: April–June 2013;
test: July–September 2013, all UTC. Do not use calibration or test targets to fit the model.

In [ ]:
from pathlib import Path
import hashlib
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

plt.rcParams.update({"figure.dpi": 100, "font.size": 12,
                     "axes.grid": True, "grid.alpha": 0.2, "svg.fonttype": "none"})
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    raise FileNotFoundError("Launch Jupyter from the repository root.")
FIGURE_DIR = ROOT / "figures" / "week4_day1"
EXPORT_ARTEFACTS = True  # Set False to display plots without exporting files.
MODEL_COLOURS = {"Persistence": "tab:orange", "Linear regression": "tab:blue",
                 "Decision tree": "tab:red", "Random forest": "tab:green"}
SLIDE_NUMBERS = {12: 31, 13: 32, 14: 34, 15: 34, 16: 35}
SPEC_SLIDE_NUMBERS = SLIDE_NUMBERS.copy()

NOTEBOOK_NAME = "tutorial_4_1b_pv_point_forecasting_solution.ipynb"
SOURCE_LABEL = "GEFCom2014 solar Zone 3; UTC"
SOURCE_SHA256 = hashlib.sha256((ROOT / "data/tutorial_4_2/gefcom2014_solar_zone3.csv").read_bytes()).hexdigest()

### Supplied plotting helpers

The next two functions keep figure styles and exports consistent. Data preparation,
model fitting and error calculations are shown in the subsequent cells.

In [ ]:
# Supplied file/plot helper. All feature and model calculations stay in later cells.
def export_figure(fig, number, slug, checkpoint, period, units, description, n):
    fig.text(0.01, 0.01, textwrap.fill(
        f"{checkpoint} | {SOURCE_LABEL} | {period} | n={n} | {units}", 118), fontsize=9)
    fig.tight_layout(rect=(0, 0.075, 1, 0.97))
    if EXPORT_ARTEFACTS:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        filename = f"w4d1_F{number:02d}_{slug}.png"
        fig.savefig(FIGURE_DIR / filename, dpi=220, bbox_inches="tight", facecolor="white")
        fig.savefig(FIGURE_DIR / filename.replace(".png", ".svg"), bbox_inches="tight", facecolor="white")
        manifest_path = ROOT / "teaching" / "week4_day1_figure_manifest.csv"
        record = pd.DataFrame([{
            "figure_id": f"F{number:02d}", "filename": f"figures/week4_day1/{filename}",
            "slide_number": SLIDE_NUMBERS[number], "spec_slide_number": SPEC_SLIDE_NUMBERS[number],
            "notebook": NOTEBOOK_NAME, "checkpoint": checkpoint, "data_period": period,
            "units": units, "description": description, "status": "generated",
            "optional": number == 17, "sample_count": n, "source_sha256": SOURCE_SHA256,
        }])
        old = pd.read_csv(manifest_path) if manifest_path.exists() else pd.DataFrame()
        if not old.empty:
            old = old.loc[old.figure_id != f"F{number:02d}"]
        pd.concat([old, record], ignore_index=True).sort_values("figure_id").to_csv(manifest_path, index=False)
    plt.show()
    plt.close(fig)

In [ ]:
def draw_table(ax, table, widths=None, fontsize=11):
    ax.axis("off")
    table = table.copy()
    for column in table.select_dtypes(include="floating"):
        table[column] = table[column].map(lambda value: f"{value:.3f}" if np.isfinite(value) else "undefined")
    artist = ax.table(cellText=table.to_numpy(), colLabels=table.columns,
                      loc="center", cellLoc="center", colWidths=widths)
    artist.auto_set_font_size(False)
    artist.set_fontsize(fontsize)
    artist.scale(1, 1.8)
    for (row, column), cell in artist.get_celld().items():
        cell.set_edgecolor("0.85")
        if row == 0:
            cell.set_facecolor("#24476a")
            cell.set_text_props(color="white", weight="bold")
    return artist

In [ ]:
PV_PATH = ROOT / "data" / "tutorial_4_2" / "gefcom2014_solar_zone3.csv"
pv_source = pd.read_csv(PV_PATH, index_col="timestamp_utc")
pv_source.index = pd.to_datetime(pv_source.index, utc=True)
pv_source = pv_source.sort_index()
pv_source["forecast_issue_time_utc"] = pd.to_datetime(pv_source["forecast_issue_time_utc"], utc=True)
expected_pv_index = pd.date_range(pv_source.index.min(), pv_source.index.max(), freq="h")
if not pv_source.index.is_unique or not pv_source.index.equals(expected_pv_index):
    raise ValueError("PV lag construction requires a complete, unique hourly index.")
WEATHER_COLUMNS = ["forecast_surface_solar_radiation_w_m2", "forecast_air_temperature_c",
                   "forecast_total_cloud_cover_fraction"]
# Explicit mapping: these descriptive source names are already in the repository CSV.
pv = pv_source[["solar_power_pu", *WEATHER_COLUMNS,
                "forecast_issue_time_utc"]].copy()
pv["solar_power_pu_raw"] = pv["solar_power_pu"].copy()
pv["solar_power_pu"] = pv["solar_power_pu_raw"].clip(lower=0, upper=1)
changed_targets = pv.loc[pv.solar_power_pu != pv.solar_power_pu_raw,
                         ["solar_power_pu_raw", "solar_power_pu"]]
pv["origin_time_utc"] = pv.index - pd.Timedelta(hours=1)
pv["solar_available"] = pv[WEATHER_COLUMNS[0]] > 1.0
if (pv["forecast_issue_time_utc"] > pv["origin_time_utc"]).any():
    raise ValueError("A weather forecast was issued after its PV forecast origin.")

### Reserve training, calibration and test periods

Define the three periods by target timestamp. Train the models on the first period
only. Reserve calibration data for the uncertainty analysis in Tutorial 4.2.

In [ ]:
PV_TRAIN_START, PV_TRAIN_END = "2012-04-01 01:00+00:00", "2013-03-31 23:00+00:00"
PV_CAL_START, PV_CAL_END = "2013-04-01 00:00+00:00", "2013-06-30 23:00+00:00"
PV_TEST_START, PV_TEST_END = "2013-07-01 00:00+00:00", "2013-09-30 23:00+00:00"
train_index = pv.loc[pd.Timestamp(PV_TRAIN_START):pd.Timestamp(PV_TRAIN_END)].index
calibration_index = pv.loc[pd.Timestamp(PV_CAL_START):pd.Timestamp(PV_CAL_END)].index
test_index = pv.loc[pd.Timestamp(PV_TEST_START):pd.Timestamp(PV_TEST_END)].index

print(f"Source: {len(pv_source):,} rows from {pv_source.index.min()} to {pv_source.index.max()}")
print("Missing required readings:", pv[["solar_power_pu", *WEATHER_COLUMNS]].isna().sum().to_dict())
print("Altered teaching targets:", len(changed_targets))
display(changed_targets)
print("Training/calibration/test before lag removal:", len(train_index), len(calibration_index), len(test_index))
display(pv.loc[train_index[:3], ["solar_power_pu", *WEATHER_COLUMNS]])

**Predict:** will the PV/irradiance peaks occur around 12:00 UTC? Examine the three training
days below. Then describe the relationship and one limitation of the weather forecasts.
Irradiance here is a weather forecast, not a measurement of actual radiation.

In [ ]:
profile_index = pv.loc["2012-05-10":"2012-05-12 23:00"].index
fig, axes = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True)
axes[0].plot(profile_index, pv.loc[profile_index, "solar_power_pu"], color="black", label="Observed PV (teaching target)")
axes[0].set(title="Three training days; Australian daylight need not match UTC daytime", ylabel="PV power (p.u.)")
axes[0].legend(fontsize=10)
axes[1].plot(profile_index, pv.loc[profile_index, WEATHER_COLUMNS[0]], color="tab:blue", label="Forecast irradiance")
axes[1].set(ylabel="Forecast irradiance (W/m²)", xlabel="Target time (UTC)")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M", tz=pv.index.tz))
axes[1].legend(fontsize=10)
export_figure(fig, 12, "pv_profile", "P0", "10–12 May 2012 UTC",
              "PV p.u.; forecast W/m²", "Separate time-aligned PV and forecast-radiation panels", len(profile_index))

## P1. Construct the previous-hour predictor (slide 32)

For a target at $t$, issue the forecast at $t-1$ hour. At that time, we assume the
previous PV observation is available. A weather forecast for $t$ can be used only
if it was issued by $t-1$.

We assume weather forecasts are available immediately after issue because the data
do not record publication delays. In practice, check both publication
and measurement delays.

**Task 1:** create `pv_lag_1` from the [0,1] target, using the complete hourly index.
An earlier calibration/test observation can be used at a later forecast origin.
We issue a new one-hour-ahead forecast each hour; we do not forecast a whole day
from one origin.

In [ ]:
pv_features = pv[WEATHER_COLUMNS].copy()
# TODO: add pv_lag_1 by shifting the bounded solar_power_pu target by one hour.
# Keep the column first so the input order is [pv_lag_1, *WEATHER_COLUMNS].
raise NotImplementedError("Construct the available one-hour PV lag.")

In [ ]:
PV_FEATURE_COLUMNS = ["pv_lag_1", *WEATHER_COLUMNS]
X_pv_train = pv_features.loc[train_index].dropna()
X_pv_calibration = pv_features.loc[calibration_index]
X_pv_test = pv_features.loc[test_index]
y_pv_train = pv.loc[X_pv_train.index, "solar_power_pu"]
y_pv_calibration = pv.loc[calibration_index, "solar_power_pu"]
y_pv_test = pv.loc[test_index, "solar_power_pu"]

print("Retained training/calibration/test:", len(y_pv_train), len(y_pv_calibration), len(y_pv_test))
print("One excluded initial training row; no excluded calibration/test targets.")

In [ ]:
example_targets = pd.DatetimeIndex(["2013-07-01 00:00+00:00", "2013-07-01 01:00+00:00", "2013-07-01 12:00+00:00"])
example = pv.loc[example_targets]
mapping = pd.DataFrame({
    "Target / valid UTC": example_targets.strftime("%d %b\n%H:%M"),
    "Origin UTC": example.origin_time_utc.dt.strftime("%d %b\n%H:%M").to_numpy(),
    "PV lag source UTC": (example_targets - pd.Timedelta(hours=1)).strftime("%d %b\n%H:%M"),
    "Observed lag (p.u.)": pv_features.loc[example_targets, "pv_lag_1"].round(3).to_numpy(),
    "Weather issued UTC": example.forecast_issue_time_utc.dt.strftime("%d %b\n%H:%M").to_numpy(),
})
display(mapping)
fig, axes = plt.subplots(2, 1, figsize=(13, 5.8), gridspec_kw={"height_ratios": [1, 1.2]})
first = example.iloc[0]
times = [first.forecast_issue_time_utc, first.origin_time_utc, example_targets[0]]
axes[0].plot(times, [0, 0, 0], color="0.6", marker="o", markersize=8)
for time, label, offset in zip(times, ["Weather issue\n30 Jun 00:00", "Origin + PV lag source\n30 Jun 23:00", "Target / valid time\n1 Jul 00:00"], [(0, 15), (-80, -45), (15, 15)]):
    axes[0].annotate(label, (time, 0), xytext=offset, textcoords="offset points", fontsize=10)
axes[0].set(yticks=[], ylim=(-0.6, 0.6), title="Issue ≤ origin < target; the final step is one hour")
axes[0].set_xlim(pd.Timestamp("2013-06-29 22:00", tz="UTC"), pd.Timestamp("2013-07-01 04:00", tz="UTC"))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M", tz=pv.index.tz))
draw_table(axes[1], mapping, fontsize=11)
for cell in axes[1].tables[0].get_celld().values():
    cell.set_height(0.2)
export_figure(fig, 13, "pv_origin_weather", "P1", "1 July 2013: 00:00, 01:00 and 12:00 UTC",
              "UTC timestamps; PV p.u.", "Three actual source/issue/origin/valid-time examples", 3)

## P2. Fit the models and generate forecasts (slide 33)

### Worked example: persistence

Persistence repeats the previous PV observation: $\hat y_{t\mid t-1}=y_{t-1}$.

Constrain predictions to [0,1], then set them to zero when forecast irradiance is
at most 1 W/m². Apply these steps to every forecasting method.
`solar_available` indicates whether forecast irradiance exceeds this threshold.
It estimates when solar production is possible, but can be wrong: actual PV may be
positive when forecast irradiance is low. The rule uses only forecast information,
not the target's observed PV. Retain both raw and corrected predictions.

In [ ]:
persistence_calibration_raw = X_pv_calibration["pv_lag_1"].copy()
persistence_test_raw = X_pv_test["pv_lag_1"].copy()
persistence_calibration = persistence_calibration_raw.clip(lower=0, upper=1)
persistence_test = persistence_test_raw.clip(lower=0, upper=1)
persistence_calibration.loc[~pv.loc[calibration_index, "solar_available"]] = 0.0
persistence_test.loc[~pv.loc[test_index, "solar_available"]] = 0.0

### Task 2: linear regression with weather forecasts

Predict $\hat y_t=\beta_0+\beta_1 y_{t-1}+\beta_R R_t^{\mathrm{fc}}+\beta_T T_t^{\mathrm{fc}}+\beta_C C_t^{\mathrm{fc}}$.

Here, $y_{t-1}$ is the previous PV observation, and $R_t^{\mathrm{fc}}$,
$T_t^{\mathrm{fc}}$ and $C_t^{\mathrm{fc}}$ are irradiance, temperature and cloud-cover
forecasts for the target hour. Each coefficient weights one predictor;
$\beta_0$ is the intercept. The output is in p.u.

Fit the model once using training data, then predict calibration and test targets
without refitting.

**Predict:** will weather forecasts help predict changes in PV output compared with persistence?

In [ ]:
pv_linear_model = LinearRegression()
# TODO: fit on X_pv_train[PV_FEATURE_COLUMNS]/y_pv_train.
# Predict calibration and test; create linear_calibration_raw and linear_test_raw
# as Series indexed by calibration_index and test_index. Do not refit.
raise NotImplementedError("Fit the linear PV model and generate predictions before applying bounds.")

### Task 3: apply PV prediction bounds

Constrain calibration and test forecasts to [0,1]: replace negative predictions with
zero and predictions above 1 with 1. Then set forecasts to zero where
`solar_available=False`, as for persistence.

Apply these rules to predictions only. Keep observed targets unchanged, including
positive output at times when forecast irradiance is low.

In [ ]:
# TODO: create linear_calibration and linear_test from the raw forecasts.
# Constrain forecasts to [0,1] and set them to zero when forecast irradiance is low.
raise NotImplementedError("Correct the linear forecasts consistently with persistence.")

### Task 4: fit a PV decision tree

Use the same four predictors as linear regression: previous-hour PV, target-hour
forecast irradiance, temperature and cloud cover. A decision tree learns threshold
rules and predicts the mean training output within each final region.
It can represent nonlinear relationships between weather and PV output.

Fit once on training data, without scaling. Fix maximum depth at 6 and at least four
cases per leaf before test evaluation. Apply the same [0,1] and low-irradiance
corrections used for persistence and regression.

**Predict:** when might a tree help with PV changes that a linear model misses?

Follow Task 2: fit on the training inputs/targets and predict both reserved periods.
Store the model in `pv_tree` and the indexed raw forecasts in
`tree_calibration_raw` and `tree_test_raw`. The correction lines are supplied.

In [ ]:
pv_tree = DecisionTreeRegressor(max_depth=6, min_samples_leaf=4, random_state=42)
# TODO: fit pv_tree on X_pv_train[PV_FEATURE_COLUMNS] and y_pv_train.
# Create tree_calibration_raw and tree_test_raw using predict on the matching inputs.
# Store them as Series indexed by calibration_index and test_index, respectively.
# Follow Task 2's structure; do not fit on calibration or test.
raise NotImplementedError("Fit the PV decision tree and generate aligned calibration/test predictions.")

# Supplied corrections: the same rules as Task 3.
tree_calibration = tree_calibration_raw.clip(lower=0, upper=1)
tree_test = tree_test_raw.clip(lower=0, upper=1)
tree_calibration.loc[~pv.loc[calibration_index, "solar_available"]] = 0.0
tree_test.loc[~pv.loc[test_index, "solar_available"]] = 0.0

### Task 5: fit a PV random forest

A random forest averages many trees fitted to resampled training rows.
At each split it considers a random subset of predictors. Use exactly the same
four predictors and training targets as the decision tree, without scaling.

The settings below are fixed before scoring. Retain raw predictions, then apply
the same corrections as every other method. Neither calibration nor test data are used to fit the model.

**Predict:** will averaging trees improve on a single tree in this test period?
Use the common error table rather than assuming a ranking.

Tutorial 4.2 uses linear regression, regardless of the ranking in this comparison.
The calendar-encoding example is in Part a. This PV example uses no calendar variables.

Complete the fit/predict lines using the same feature order as the tree.
Store the model in `pv_forest` and the indexed raw forecasts in
`forest_calibration_raw` and `forest_test_raw`. Use the same fitted model for both periods.

In [ ]:
pv_forest = RandomForestRegressor(n_estimators=150, min_samples_leaf=4, max_features=0.8,
                                  random_state=42, n_jobs=1)
# TODO: fit pv_forest on the SAME training predictors/targets as the PV decision tree.
# Create forest_calibration_raw and forest_test_raw as indexed Series.
# Reuse the fitted forest for both periods; do not refit.
raise NotImplementedError("Fit the PV random forest and generate aligned calibration/test predictions.")

# Supplied corrections: keep raw and corrected predictions separate.
forest_calibration = forest_calibration_raw.clip(lower=0, upper=1)
forest_test = forest_test_raw.clip(lower=0, upper=1)
forest_calibration.loc[~pv.loc[calibration_index, "solar_available"]] = 0.0
forest_test.loc[~pv.loc[test_index, "solar_available"]] = 0.0

In [ ]:
pv_forecasts = pd.DataFrame({"Persistence": persistence_test, "Linear regression": linear_test,
                              "Decision tree": tree_test, "Random forest": forest_test}, index=test_index)
raw_forecasts = pd.DataFrame({"Persistence": persistence_test_raw, "Linear regression": linear_test_raw,
                             "Decision tree": tree_test_raw, "Random forest": forest_test_raw})

correction_rows = []
for name in pv_forecasts:
    raw = raw_forecasts[name]
    correction_rows.append({"Method": name,
                            "Raw outside [0,1]": int(((raw < 0) | (raw > 1)).sum()),
                            "Changed test forecasts": int(pv_forecasts[name].ne(raw).sum())})
display(pd.DataFrame(correction_rows))

## P3. Ramps, missed peaks and forecast errors (slide 34)

**Task 6:** use the supplied functions to calculate full-test MAE and RMSE for all four methods.
Pass actuals first and forecasts second. Both metrics are in p.u.
Do not calculate PV MAPE because actual output is often zero.

**Predict:** could many zero-output hours make full-day errors appear small?

The supplied second table evaluates the same models when forecast irradiance exceeds
1 W/m². Report both subsets with their counts. This subset uses forecast information,
not observed PV, and is not a perfectly known daylight mask.

The weekly plot is enlarged around the largest observed one-hour change in that
displayed week. Selection is solely for diagnosis, not for model fitting or tuning.
Compare the timing and magnitude of the ramp. What does a point prediction leave unspecified?

In [ ]:
def metric_arrays(actual, forecast):
    '''Check paired Series before converting them to numerical arrays.'''
    if not isinstance(actual, pd.Series) or not isinstance(forecast, pd.Series):
        raise TypeError("Pass actual and forecast as pandas Series.")
    if len(actual) == 0 or len(actual) != len(forecast) or not actual.index.equals(forecast.index):
        raise ValueError("Actual and forecast must have equal, non-empty aligned timestamps.")
    if not actual.index.is_unique:
        raise ValueError("Metric timestamps must be unique.")
    a, f = actual.to_numpy(dtype=float), forecast.to_numpy(dtype=float)
    if not np.isfinite(a).all() or not np.isfinite(f).all():
        raise ValueError("Resolve/count non-finite inputs upstream; metrics never drop pairs.")
    return a, f

def mae(actual, forecast):
    a, f = metric_arrays(actual, forecast)
    return float(np.mean(np.abs(a - f)))

def rmse(actual, forecast):
    a, f = metric_arrays(actual, forecast)
    return float(np.sqrt(np.mean((a - f) ** 2)))

In [ ]:
# TODO: create pv_comparison with models as rows and MAE (p.u.), RMSE (p.u.), n.
# Use the complete common y_pv_test and each corrected pv_forecasts column.
raise NotImplementedError("Compare all four models on the same test targets.")

In [ ]:
solar_subset = pv.loc[test_index, "solar_available"]
pv_subset_rows = []
for name in pv_forecasts:
    # Reuse the all-hour student calculation. Compute the disclosed secondary subset.
    pv_subset_rows.append({"Subset": "All hours", "Model": name,
                          "MAE (p.u.)": pv_comparison.loc[name, "MAE (p.u.)"],
                          "RMSE (p.u.)": pv_comparison.loc[name, "RMSE (p.u.)"], "n": len(y_pv_test)})
    pv_subset_rows.append({"Subset": "Forecast irradiance >1 W/m²", "Model": name,
                          "MAE (p.u.)": mae(y_pv_test.loc[solar_subset], pv_forecasts.loc[solar_subset, name]),
                          "RMSE (p.u.)": rmse(y_pv_test.loc[solar_subset], pv_forecasts.loc[solar_subset, name]),
                          "n": int(solar_subset.sum())})
pv_subset_comparison = pd.DataFrame(pv_subset_rows)
display(pv_subset_comparison.round(3))

In [ ]:
pv_plot_index = y_pv_test.loc["2013-07-15":"2013-07-21 23:00"].index
# Select the largest observed one-hour change WITHIN the displayed week.
week_changes = y_pv_test.loc[pv_plot_index].diff().abs()
ramp_time = week_changes.idxmax()
ramp_index = y_pv_test.loc[ramp_time - pd.Timedelta(hours=6):ramp_time + pd.Timedelta(hours=6)].index
fig, axes = plt.subplots(2, 1, figsize=(13, 7.5), gridspec_kw={"height_ratios": [1, 1.1]})
for ax, index in zip(axes, [pv_plot_index, ramp_index]):
    ax.plot(index, y_pv_test.loc[index], color="black", linewidth=2, label="Actual PV")
    for name in pv_forecasts:
        ax.plot(index, pv_forecasts.loc[index, name], color=MODEL_COLOURS[name],
                linewidth=1.3, marker="o" if ax == axes[1] else None, markersize=4, label=name)
    ax.set(ylabel="PV power (p.u.)", xlabel="Target time (UTC)", ylim=(-0.03, 1.03))
    ax.legend(ncol=3, fontsize=9)
axes[0].set_title("One-hour-ahead forecasts: weekly overview")
axes[0].axvspan(ramp_index.min(), ramp_index.max(), color="0.5", alpha=0.15, label="Enlarged below")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%d %b", tz=pv.index.tz))
axes[1].set_title(f"Largest observed hourly change in display week: {ramp_time:%d %b %Y %H:%M} UTC")
axes[1].axvline(ramp_time, color="0.5", linestyle="--")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M", tz=pv.index.tz))
print("Ramp change:", y_pv_test.diff().loc[ramp_time], "p.u.; target:", ramp_time)
export_figure(fig, 14, "pv_forecasts", "P3", "15–21 July 2013 UTC; largest-ramp close-up",
              "PV p.u.", "Weekly overview and observed ramp selected transparently for diagnosis", len(pv_plot_index))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title("Point forecast scores: all test hours and forecast-solar-available hours", pad=18)
draw_table(ax, pv_subset_comparison.round(3), widths=[0.33, 0.27, 0.16, 0.16, 0.08], fontsize=11)
export_figure(fig, 15, "pv_scores", "P3", "1 July–30 September 2013 UTC",
              "MAE/RMSE p.u.", "All hours versus forecast irradiance >1 W/m²; common targets and counts", len(y_pv_test))

## P4. Calibration and forecast uncertainty (slide 35)

The diagram summarises the three chronological periods. The linear model is trained
once, then used to predict the calibration and test targets.

Tutorial 4.2 constructs prediction intervals from calibration errors:
**actual minus the corrected forecast**. It rebuilds this model directly from source
data, so no saved model, prediction bundle or previous notebook state is required.
Do not calculate intervals or coverage in this point-forecasting tutorial.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.7))
periods = [("Training", PV_TRAIN_START, PV_TRAIN_END, len(y_pv_train), "tab:blue"),
           ("Calibration", PV_CAL_START, PV_CAL_END, len(y_pv_calibration), "tab:green"),
           ("Test", PV_TEST_START, PV_TEST_END, len(y_pv_test), "tab:orange")]
for row, (label, start, end, count, colour) in enumerate(periods):
    left = mdates.date2num(pd.Timestamp(start))
    right = mdates.date2num(pd.Timestamp(end) + pd.Timedelta(hours=1))
    ax.barh(row, right - left, left=left, height=0.55, color=colour)
ax.set(yticks=range(3), yticklabels=[f"{label}: {count:,} targets" for label, _, _, count, _ in periods],
       title="Fit linear regression once; reuse unchanged for calibration and test", xlabel="Target date (UTC)")
ax.text(0.01, 0.93, "Train: Apr 2012–Mar 2013; calibration: Apr–Jun 2013; test: Jul–Sep 2013",
        transform=ax.transAxes, fontsize=10)
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y", tz=pv.index.tz))
export_figure(fig, 16, "pv_split_handover", "P4", "1 April 2012–30 September 2013 UTC",
              "timestamps; target counts", "Fixed linear model; reserved calibration; no intervals yet", len(y_pv_train) + len(y_pv_calibration) + len(y_pv_test))

In [ ]:
# Optional instructor exports; Tutorial 4.2 does not load these files.
if EXPORT_ARTEFACTS:
    result_dir = ROOT / "teaching/week4_day1_outputs"
    result_dir.mkdir(parents=True, exist_ok=True)
    pv_forecasts.to_csv(result_dir / "pv_test_forecasts.csv")
    pv_comparison.to_csv(result_dir / "pv_scores.csv")
    pv_subset_comparison.to_csv(result_dir / "pv_subset_scores.csv", index=False)

**Exit question:** why can PV lag 1 be used at every hourly origin, while observed load
lag 1 is unavailable for most targets at a common day-ahead origin?